In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('tf').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/27 19:19:59 WARN Utils: Your hostname, developer, resolves to a loopback address: 127.0.1.1; using 192.168.2.197 instead (on interface wlx00e02d972611)
25/08/27 19:19:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/27 19:20:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/27 19:20:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/08/27 19:20:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/08/27 19:20:05 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [12]:
df = spark.read.csv('data.csv', header=True, inferSchema=True)

In [13]:
df.show()

+----------+------------------+-----------+----------+-----+-------+--------+--------------+--------+--------+
|      Date|      Product Name|   Category|Units Sold|Price|Revenue|Discount|Units Returned|Location|Platform|
+----------+------------------+-----------+----------+-----+-------+--------+--------------+--------+--------+
|2020-01-06|      Whey Protein|    Protein|       143|31.98|4573.14|    0.03|             2|  Canada| Walmart|
|2020-01-06|         Vitamin C|    Vitamin|       139|42.51|5908.89|    0.04|             0|      UK|  Amazon|
|2020-01-06|          Fish Oil|      Omega|       161|12.91|2078.51|    0.25|             0|  Canada|  Amazon|
|2020-01-06|      Multivitamin|    Vitamin|       140|16.07| 2249.8|    0.08|             0|  Canada| Walmart|
|2020-01-06|       Pre-Workout|Performance|       157|35.47|5568.79|    0.25|             3|  Canada|   iHerb|
|2020-01-06|              BCAA| Amino Acid|       154|41.19|6343.26|    0.13|             1|      UK| Walmart|
|

In [14]:
df_transformed = df.filter(df['Units Sold']>5).groupBy('Category').sum()

In [16]:
df_transformed.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Category#204], functions=[sum(Units Sold#205), sum(Price#206), sum(Revenue#207), sum(Discount#208), sum(Units Returned#209)])
   +- Exchange hashpartitioning(Category#204, 200), ENSURE_REQUIREMENTS, [plan_id=129]
      +- HashAggregate(keys=[Category#204], functions=[partial_sum(Units Sold#205), partial_sum(Price#206), partial_sum(Revenue#207), partial_sum(Discount#208), partial_sum(Units Returned#209)])
         +- Filter (isnotnull(Units Sold#205) AND (Units Sold#205 > 5))
            +- FileScan csv [Category#204,Units Sold#205,Price#206,Revenue#207,Discount#208,Units Returned#209] Batched: false, DataFilters: [isnotnull(Units Sold#205), (Units Sold#205 > 5)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/developer/pyspark/data.csv], PartitionFilters: [], PushedFilters: [IsNotNull(Units Sold), GreaterThan(Units Sold,5)], ReadSchema: struct<Category:string,Units Sold:int,Price:double,Revenue